nn

In [2]:
import pandas as pd
import numpy as np
import matplotlib_inline as plt
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("Data/emails.csv")
print(df.shape)
print(df.head())
print(df.columns)

(5728, 2)
                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1
Index(['text', 'spam'], dtype='str')


In [4]:

# 1. Dataset overview
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

# 2. Missing values
print("\nMissing values:")
print(df.isnull().sum())

# 3. Class distribution
print("\nSpam label distribution:")
print(df["spam"].value_counts())

print("\nClass proportions (%):")
print((df["spam"].value_counts(normalize=True) * 100).round(2))

# 4. Duplicate emails
print("\nDuplicate rows:", df.duplicated().sum())

# 5. Sample emails from each class
print("\nSpam example:")
print(df.loc[df["spam"] == 1, "text"].iloc[0][:500])

print("\nNon-spam example:")
print(df.loc[df["spam"] == 0, "text"].iloc[0][:500])


Shape: (5728, 2)

Data types:
text      str
spam    int64
dtype: object

Missing values:
text    0
spam    0
dtype: int64

Spam label distribution:
spam
0    4360
1    1368
Name: count, dtype: int64

Class proportions (%):
spam
0    76.12
1    23.88
Name: proportion, dtype: float64

Duplicate rows: 33

Spam example:
Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the  market is full of suqgestions and the information isoverwhelminq ; but a good  catchy logo , stylish statlonery and outstanding website  will make the task much easier .  we do not promise that havinq ordered a iogo your  company will automaticaily become a world ieader : it isguite ciear that  without good products , effective business organization and practicable aim it  will be hotat nowad

Non-spam example:
Subject: hello guys ,  i ' m " bugging you " for your completed questionnaire and for a one - page  bio / statement on your thoughts on " business edu and the new

In [5]:
data = df.copy()

print("Rows before cleaning:", len(data))
data = data.drop_duplicates().reset_index(drop=True)
print("Rows after cleaning:", len(data))
print("Duplicate rows removed:", len(df)-len(data))

Rows before cleaning: 5728
Rows after cleaning: 5695
Duplicate rows removed: 33


In [6]:
print("\nlabel distribution:")
print(data["spam"].value_counts())


label distribution:
spam
0    4327
1    1368
Name: count, dtype: int64


In [7]:
print("\nEmpty email texts:", (data["text"].str.strip() == "").sum())


Empty email texts: 0


In [8]:
X = data["text"]
y = data["spam"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)



X shape: (5695,)
y shape: (5695,)


In [9]:
# train test splitting

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

In [10]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4556
Testing samples: 1139


In [11]:
print(y_train.value_counts(normalize=True).round(3))
print(y_test.value_counts(normalize=True).round(3))

spam
0    0.76
1    0.24
Name: proportion, dtype: float64
spam
0    0.759
1    0.241
Name: proportion, dtype: float64


In [12]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    lowercase=True,
    max_features=10000
)

# Learn vocabulary and transform training emails
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test emails using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


Training matrix shape: (4556, 10000)
Testing matrix shape: (1139, 10000)
Vocabulary size: 10000


In [14]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

print("Model training complete!")

Model training complete!


In [15]:

y_pred = nb_model.predict(X_test_tfidf)

print("Actual labels:    ", y_test.iloc[:20].tolist())
print("Predicted labels: ", y_pred[:20].tolist())


Actual labels:     [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]
Predicted labels:  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]


In [16]:


from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Spam", "Spam"]
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.9622

Classification report:
              precision    recall  f1-score   support

    Not Spam       0.95      1.00      0.98       865
        Spam       1.00      0.84      0.91       274

    accuracy                           0.96      1139
   macro avg       0.98      0.92      0.95      1139
weighted avg       0.96      0.96      0.96      1139


Confusion matrix:
[[865   0]
 [ 43 231]]


In [17]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score

# Build a complete text-classification pipeline
lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=10000
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train on raw training text; TF-IDF is fitted only on training data
lr_model.fit(X_train, y_train)

# Evaluate on the same held-out test set
lr_pred = lr_model.predict(X_test)

print("Logistic Regression accuracy:",
      round(accuracy_score(y_test, lr_pred), 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    lr_pred,
    target_names=["Not Spam", "Spam"]
))


Logistic Regression accuracy: 0.9868

Classification report:
              precision    recall  f1-score   support

    Not Spam       0.98      1.00      0.99       865
        Spam       1.00      0.95      0.97       274

    accuracy                           0.99      1139
   macro avg       0.99      0.97      0.98      1139
weighted avg       0.99      0.99      0.99      1139



In [18]:

# Our selected model: TF-IDF + Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=10000
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train on raw training emails
final_model.fit(X_train, y_train)

print("Final model trained successfully!")


Final model trained successfully!


In [19]:

from sklearn.metrics import accuracy_score, classification_report

final_pred = final_model.predict(X_test)

print("Final accuracy:", round(accuracy_score(y_test, final_pred), 4))
print("\n", classification_report(
    y_test,
    final_pred,
    target_names=["Not Spam", "Spam"]
))


Final accuracy: 0.9868

               precision    recall  f1-score   support

    Not Spam       0.98      1.00      0.99       865
        Spam       1.00      0.95      0.97       274

    accuracy                           0.99      1139
   macro avg       0.99      0.97      0.98      1139
weighted avg       0.99      0.99      0.99      1139



In [20]:

email = """
Subject: Congratulations! You have won a free prize.
Click here to claim your reward now!
"""

prediction = final_model.predict([email])[0]

if prediction == 1:
    print("🚨 Spam email detected!")
else:
    print("✅ This email is classified as not spam.")


🚨 Spam email detected!


In [21]:

import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(final_model, "models/spam_classifier.pkl")

print("Model saved to models/spam_classifier.pkl")


Model saved to models/spam_classifier.pkl


In [22]:

# Check whether identical email text appears in both splits

train_emails = set(X_train.astype(str))
test_emails = set(X_test.astype(str))

overlap = train_emails.intersection(test_emails)

print("Unique training emails:", len(train_emails))
print("Unique testing emails:", len(test_emails))
print("Overlapping email texts:", len(overlap))


Unique training emails: 4556
Unique testing emails: 1139
Overlapping email texts: 0
